Required Python Packages Installation

Run the following command if the required packages are not already installed:

In [ ]:
#!pip install pandas

In [1]:
import os
import pandas as pd
import logging

### 1. Helper Function: clean_id():

Standardizes spatial ID numbers so they match perfectly across datasets.

- **What it does:** If an ID gets converted into a float (e.g., 316700.0), this function converts it back into a clean string ("316700"). It also removes accidental surrounding white space.

In [ ]:
def clean_id(val):
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

### Quality Control: verify_pipeline_handoff().

Performs automated safety checks before exporting the data

- **What it does:** Uses assert statements to confirm essential columns exist and validates that listing prices are positive. If a check fails, the pipeline halts immediately to prevent saving corrupted output.


In [2]:
def verify_pipeline_handoff(raw_df: pd.DataFrame, final_df: pd.DataFrame) -> None:
    # 1. Feature Completeness Check
    required_cols = ["id", "price", "sa2_code", "area_name", "TimeFrame_dt", "price_gap"]
    for col in required_cols:
        assert col in final_df.columns, f"❌ Pipeline Failure: Essential metric '{col}' missing!"

    # 2. Business Logic Boundaries (Prices must be positive)
    invalid_prices = (final_df["price"] <= 0).sum()
    assert invalid_prices == 0, f"❌ Sanity Violation: Found {invalid_prices} records with price <= 0!"

### 2. Complete End-to-End Pipeline Execution (run_pipeline)

Here is how all components (Ingestion $\rightarrow$ API Enrichment $\rightarrow$ Merging $\rightarrow$ In-Pipeline Sanity Check $\rightarrow$ Export) fit together cleanly:

In [ ]:
def clean_id(val):
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s

def run_pipeline(airbnb_csv_path: str, bond_csv_path: str, output_dir: str):
    logging.info("Starting Pipeline Execution...")

    # Step 1: Ingestion
    raw_airbnb = pd.read_csv(airbnb_csv_path)
    raw_bond = pd.read_csv(bond_csv_path)

    # Clean header whitespace
    raw_airbnb.columns = raw_airbnb.columns.str.strip()
    raw_bond.columns = raw_bond.columns.str.strip()

    # Step 2: Key Standardisation & Date Parsing
    # Apply clean_id to standardise float-string anomalies (e.g., '316700.0' -> '316700')
    raw_airbnb['sa2_code'] = raw_airbnb['sa2_code'].apply(clean_id)
    raw_bond['Location Id'] = raw_bond['Location Id'].apply(clean_id)

    raw_airbnb['TimeFrame_dt'] = pd.to_datetime(raw_airbnb['TimeFrame'], dayfirst=True, errors='coerce')
    raw_bond['TimeFrame_dt'] = pd.to_datetime(raw_bond['TimeFrame'], dayfirst=True, errors='coerce')

    # Filter bond dataset to summary rows only ('ALL' Dwelling & 'ALL' Bedrooms)
    bond_summary = raw_bond[
        (raw_bond['Dwelling Type'].astype(str).str.upper() == 'ALL') & 
        (raw_bond['Number Of Beds'].astype(str).str.upper() == 'ALL')
    ].copy()

    # Step 3: Dataset Joining (1-to-1 aggregate join)
    merged_df = raw_airbnb.merge(
        bond_summary,
        left_on=["sa2_code", "TimeFrame_dt"],
        right_on=["Location Id", "TimeFrame_dt"],
        how="inner"
    )

    # Step 4: Economic Metric Calculations
    merged_df["daily_bond_rent"] = merged_df["Median Rent"] / 7.0
    merged_df["price_gap"] = merged_df["price"] - merged_df["daily_bond_rent"]

    # Step 5: In-Pipeline Sanity Verification
    verify_pipeline_handoff(raw_airbnb, merged_df)
    
    merged_df.shape

    # Step 6: Safe Export
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "christchurch_airbnb_with_sa2.csv")
    if os.path.exists(output_path):
        os.remove(output_path)

    merged_df.to_csv(output_path, index=False)
    logging.info(f"Pipeline finished successfully. Output saved to: {output_path}")

    return merged_df  # Return dataframe to notebook environment

# Execute Pipeline
Dir_path = "../../Data_Wrangling_Datasets"

# Run pipeline and save output to variable
if __name__ == "__main__":
    merged_df = run_pipeline(
        airbnb_csv_path=os.path.join(
            Dir_path, "christchurch_airbnb_with_sa2.csv"
        ),
        bond_csv_path=os.path.join(Dir_path, "cleaned_bond_data_aligned.csv"),
        output_dir="./output",
    )

    # Now you can inspect shape in any notebook cell:
    print(merged_df.shape)

2026-09-29 19:34:54,659 [INFO] Starting Pipeline Execution...


KeyError: 'LocationId'